# <h1 style="text-align:center">***Gaussian Naive Bayes from Scratch***</h1>
#### Theoretical Foundations of Machine Learning —  **Assignment one**

**Members:**
- Ahmed Mohamed Elsayed Tolba — 20242023
- Mohamed Faisal Mohamed Salem — 20240523

**Dataset:** Breast Cancer Wisconsin

---

## Task 1: Data Preparation

In this task we load the Breast Cancer Wisconsin dataset, inspect its shape and class distribution, and split it into training and test sets.
<br> The split must be stratified, 80% training and 20% testing, with seed 42, so each class keeps the same proportion in both sets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
x = data.data
y = data.target
print("="*80)
print("X shape:", x.shape)
print("y shape:", y.shape)
print("Feature names:", data.feature_names[:5])
print("Target names:", data.target_names)
print("Class counts:", np.unique(y, return_counts=True))
print("="*80)
def train_test_split(x, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_test = int(len(idx) * test_size)
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return x[train_idx], x[test_idx], y[train_idx], y[test_idx]

x_train, x_test, y_train, y_test = train_test_split(x, y)
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Train class counts:", np.unique(y_train, return_counts=True))
print("Test class counts:", np.unique(y_test, return_counts=True))
print("="*80)

train_mean = x_train.mean(axis=0)
train_std = x_train.std(axis=0)
train_std[train_std == 0] = 1
x_train_scaled = (x_train - train_mean) / train_std
x_test_scaled = (x_test - train_mean) / train_std

print("STANDARDIZATION RESULTS")
print("Original training shape:", x_train.shape)
print("Original testing shape:", x_test.shape)
print("Mean of first 5 scaled features:",x_train_scaled.mean(axis=0)[:5])
print("Standard deviation of first 5 scaled features:",x_train_scaled.std(axis=0)[:5])
print("=" * 80)


X shape: (569, 30)
y shape: (569,)
Feature names: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness']
Target names: ['malignant' 'benign']
Class counts: (array([0, 1]), array([212, 357]))
x_train shape: (456, 30)
x_test shape: (113, 30)
y_train shape: (456,)
y_test shape: (113,)
Train class counts: (array([0, 1]), array([170, 286]))
Test class counts: (array([0, 1]), array([42, 71]))
STANDARDIZATION RESULTS
Original training shape: (456, 30)
Original testing shape: (113, 30)
Scaled training shape: (456, 30)
Mean of first 5 scaled features: [-4.36541641e-15  1.27006105e-15  2.53793088e-15  9.15447055e-16
 -4.78369780e-15]
Standard deviation of first 5 scaled features: [1. 1. 1. 1. 1.]


### Does GNB need feature scaling?

No. Accuracy without scaling was 0.938 (106/113) and accuracy with scaling was 0.929 (105/113), a difference of only 0.009, or 1 test sample. This is because GNB estimates a separate mean and variance for each feature in each class, so rescaling a feature rescales its Gaussian parameters in the same way for both classes and does not change which class has the larger log score in any important way.

---

## Task 2: Gaussian Naive Bayes Implementation

We implement the classifier from scratch. Training means estimating, for each class and each feature, a prior, a mean and a variance. Prediction means computing,
<br> for each test sample and each class, a log score, and picking the class with the largest score. We work in log space to avoid underflow.

In [ ]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, x, y):
        n_samples, n_features = x.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        self.theta = np.zeros((n_classes, n_features))
        self.var = np.zeros((n_classes, n_features))
        self.class_prior = np.zeros(n_classes)

        for i, c in enumerate(self.classes):
            x_c = x[y == c]
            self.theta[i] = x_c.mean(axis=0)
            self.var[i] = x_c.var(axis=0)
            self.class_prior[i] = x_c.shape[0] / n_samples

        epsilon = self.var_smoothing * np.max(self.var)
        self.var = self.var + epsilon
        return self
    
    def _joint_log_likelihood(self, x):
        log_prior = np.log(self.class_prior)
        diff = x[:, None, :] - self.theta
        log_prob = -0.5 * np.log(2 * np.pi * self.var) - (diff ** 2) / (2 * self.var)
        return log_prior + log_prob.sum(axis=2)
    
    def predict(self, x):
        log_joint = self._joint_log_likelihood(x)
        return self.classes[np.argmax(log_joint, axis=1)]
    
    def predict_proba(self, x):
        log_joint = self._joint_log_likelihood(x)
        log_joint = log_joint - log_joint.max(axis=1, keepdims=True)
        proba = np.exp(log_joint)
        return proba / proba.sum(axis=1, keepdims=True)

model = GaussianNaiveBayes()
model.fit(X_train, y_train)
print("classes_:", model.classes)
print("class_prior_:", model.class_prior)
print("theta_ shape:", model.theta.shape)
print("var_ shape:", model.var.shape)
print("theta for malignant, first 3 features:", model.theta[0, :3])

y_pred = model.predict(x_test)
print("y_pred first 10:", y_pred[:10])
print("y_test first 10:", y_test[:10])
print("Accuracy (quick check):", (y_pred == y_test).mean())

classes_: [0 1]
class_prior_: [0.37280702 0.62719298]
theta_ shape: (2, 30)
var_ shape: (2, 30)
theta for malignant, first 3 features: [ 17.366       21.67541176 114.65817647]
y_pred first 10: [0 0 1 1 1 1 0 1 1 1]
y_test first 10: [0 0 1 1 1 1 0 1 1 1]
Accuracy (quick check): 0.9380530973451328


### Task one question code 


In [47]:
model_unscaled = GaussianNaiveBayes()
model_unscaled.fit(x_train, y_train)
acc_unscaled = (model_unscaled.predict(x_test) == y_test).mean()

model_scaled = GaussianNaiveBayes()
model_scaled.fit(x_train_scaled, y_train)
acc_scaled = (model_scaled.predict(x_test_scaled) == y_test).mean()

print("Accuracy without scaling:", acc_unscaled)
print("Accuracy with scaling:", acc_scaled)

Accuracy without scaling: 0.9380530973451328
Accuracy with scaling: 0.9292035398230089


### Sanity Test

We test the classifier on a tiny dataset we made by hand, with 2 classes and 1 feature, so we can calculate the mean, variance and one posterior with a calculator and check that our class gives the same numbers.

In [48]:
x_sanity = np.array([[1], [2], [3], [5], [6], [7]])
y_sanity = np.array([0, 0, 0, 1, 1, 1])

tiny_model = GaussianNaiveBayes()
tiny_model.fit(x_sanity, y_sanity)

print("classes:", tiny_model.classes)
print("class_prior:", tiny_model.class_prior)
print("theta:", tiny_model.theta)
print("var:", tiny_model.var)
print("Prediction for 2.5:", tiny_model.predict(np.array([[2.5]])))
print("Probabilities for 2.5:", tiny_model.predict_proba(np.array([[2.5]])))

classes: [0 1]
class_prior: [0.5 0.5]
theta: [[2.]
 [6.]]
var: [[0.66666667]
 [0.66666667]]
Prediction for 2.5: [0]
Probabilities for 2.5: [[9.99876605e-01 1.23394577e-04]]
